<table style="width:100%; background-color: white; padding: 10px; border-radius: 6px; box-shadow: 0 0 5px rgba(0,0,0,0.2);">
  <tr>
    <td>
      <h1 style="margin-bottom: 0; color: black; font-size: clamp(1.5rem, 2.5vw, 2.5rem);">
        Drone RL with LLM-Guided Curriculum Learning
      </h1>
      <p style="margin-top: 8px; color: black; font-size: 1rem;">
        EVA Deep Reinforcement Learning &ndash; DRL Project, FS 2026<br>
        Rino M. Albertin  
      </p>
    </td>
    <td align="right">
      <img src="docs/images/OST_Logo_DE_RGB@2000ppi.png" alt="OST Logo" width="180">
    </td>
  </tr>
</table>

## Executive Summary

This project studies single-drone trajectory tracking in simulation with reinforcement learning and curriculum learning. The implemented control learner is a PPO policy trained through Stable-Baselines3 on a compact trajectory-tracking environment built around `gym-pybullet-drones` and PyBullet.

The LLM is not a drone controller. Its role in the implemented Phase 6 pipeline is a curriculum/task proposer: it suggests structured trajectory tasks, while deterministic validation decides whether those tasks are safe and feasible enough to enter training.

On this branch, direct PPO training, fixed/manual curriculum training, configurable single- and multi-environment PPO rollout collection, config-driven evaluation suites, and the Phase 6 LLM curriculum proposal/training pipeline are implemented. The notebook does not claim LLM curriculum performance unless real saved result artifacts are present locally. The Phase 5 cross-run comparison pipeline remains future work.


## Introduction

The task is single-drone reference trajectory tracking in a physics simulator. A policy observes the current drone position, the current reference position, the tracking error, and trajectory progress, then chooses continuous target-position actions that are mapped into the simulator's PID action bounds.

Curriculum learning is useful here because tracking difficult paths immediately can produce poor exploration, early truncation, or policies that learn to hover instead of following the reference. A fixed manual curriculum provides a controlled baseline: it starts with hover and short-line tasks before moving to longer line tracking. The implemented LLM curriculum pipeline adds an adaptive path where recent accepted tasks, rejected proposals, and compact metrics can be used as context for the next JSON task proposal.

Multi-drone scenes are optional showcase material only in this project scope. The implemented learning problem is single-drone tracking, not full multi-agent reinforcement learning.


## Objective and Approach

**Research question:** Can an LLM propose valid and useful training tasks that improve drone RL learning compared with direct training and a fixed manual curriculum?

The intended comparison is:

1. Direct PPO training on the target task.
2. Fixed/manual curriculum PPO training.
3. LLM-guided adaptive curriculum PPO training.

The current branch supports all three workflow mechanics. Direct PPO and manual curriculum are configured for smoke, medium, and final tiers. Phase 6 now implements strict JSON LLM proposals, deterministic validation, bounded repair attempts, JSONL proposal logging, dry-run proposal checks, and stage-wise PPO training. Final comparative claims still require report-ready saved runs and the Phase 5 comparison pipeline, which is not implemented yet.


## Repository and Runtime Overview


In [ ]:
from __future__ import annotations

import json
import os
import platform
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path
from typing import Any

TRAIN_FROM_SCRATCH = False
RUN_QUICK_DEMO = True
USE_SAVED_RESULTS = True


def find_repo_root(start: Path | None = None) -> Path:
    """Resolve the repository root from the current notebook location."""
    current = (start or Path.cwd()).expanduser().resolve(strict=False)
    candidates = (current, *current.parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    message = "Could not find repository root containing pyproject.toml and src/."
    raise RuntimeError(message)


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import envs, evaluation, experiments, llm, trajectories, utils, validation  # noqa: E402

PUBLIC_ALIASES = {
    "envs": envs.__name__,
    "evaluation": evaluation.__name__,
    "experiments": experiments.__name__,
    "llm": llm.__name__,
    "trajectories": trajectories.__name__,
    "utils": utils.__name__,
    "validation": validation.__name__,
}
print("Public package aliases imported:", ", ".join(PUBLIC_ALIASES))


def unique_paths(paths: list[Path]) -> list[Path]:
    """Return paths with duplicates removed while preserving order."""
    seen: set[str] = set()
    unique: list[Path] = []
    for item in paths:
        key = str(item.expanduser().resolve(strict=False))
        if key not in seen:
            seen.add(key)
            unique.append(item.expanduser().resolve(strict=False))
    return unique


def storage_root_candidates() -> list[Path]:
    """Return storage roots that may hold local generated run artifacts."""
    candidates: list[Path] = []
    env_storage = os.environ.get("STORAGE_ROOT")
    if env_storage:
        candidates.append(Path(env_storage))
    try:
        candidates.append(utils.paths.get_storage_root())
    except Exception as exc:  # noqa: BLE001 - report notebook should not crash on path helper issues.
        print(f"Could not resolve utils.paths.get_storage_root(): {exc}")
    candidates.extend(
        [
            REPO_ROOT / "storage",
            REPO_ROOT.parent / "storage",
            Path("/workspace/storage"),
        ]
    )
    return unique_paths(candidates)


STORAGE_CANDIDATES = storage_root_candidates()
STORAGE_ROOT = next(
    (candidate for candidate in STORAGE_CANDIDATES if (candidate / "runs").is_dir()),
    STORAGE_CANDIDATES[0],
)
RUNS_ROOT = STORAGE_ROOT / "runs"

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"STORAGE_ROOT selected: {STORAGE_ROOT}")
print("Storage candidates:")
for candidate in STORAGE_CANDIDATES:
    marker = "selected" if candidate == STORAGE_ROOT else "candidate"
    runs_state = "runs/ exists" if (candidate / "runs").is_dir() else "runs/ missing"
    print(f"  - {candidate} [{marker}, {runs_state}]")
print(f"Flags: TRAIN_FROM_SCRATCH={TRAIN_FROM_SCRATCH}, RUN_QUICK_DEMO={RUN_QUICK_DEMO}, USE_SAVED_RESULTS={USE_SAVED_RESULTS}")

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import Image
    from IPython.display import display as ipython_display
except ImportError:
    Image = None
    ipython_display = print

try:
    import yaml
except ImportError:
    yaml = None


def show_table(rows: list[dict[str, Any]], columns: list[str] | None = None, title: str | None = None) -> None:
    """Display rows as a compact table when pandas/IPython is available."""
    if title:
        print(title)
    if not rows:
        print("No rows available.")
        return
    selected_rows = rows
    if columns is not None:
        selected_rows = [{column: row.get(column) for column in columns} for row in rows]
    if pd is not None:
        ipython_display(pd.DataFrame(selected_rows))
    else:
        print(json.dumps(selected_rows, indent=2, sort_keys=True))


def load_yaml(path: Path) -> dict[str, Any]:
    """Load a YAML mapping with a clear error if PyYAML is unavailable."""
    if yaml is None:
        message = "PyYAML is not available in this environment."
        raise RuntimeError(message)
    payload = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        message = f"{path} did not contain a YAML mapping."
        raise TypeError(message)
    return payload


def git_output(*args: str) -> str:
    """Run a read-only git command and return compact text."""
    try:
        completed = subprocess.run(  # noqa: S603 - fixed git executable with read-only args in a report notebook.
            ["git", *args],  # noqa: S607 - git is expected on PATH in this development container.
            cwd=REPO_ROOT,
            check=True,
            capture_output=True,
            text=True,
        )
    except Exception as exc:  # noqa: BLE001 - notebook report should remain executable outside Git.
        return f"unavailable: {exc}"
    return completed.stdout.strip() or "<clean/no output>"


print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Git branch:", git_output("branch", "--show-current"))
print("Git status --short:", git_output("status", "--short"))

In [ ]:
DEPENDENCY_NAMES = [
    "numpy",
    "pandas",
    "matplotlib",
    "yaml",
    "gymnasium",
    "stable_baselines3",
    "gym_pybullet_drones",
    "torch",
    "wandb",
]

runtime_rows: list[dict[str, Any]] = []
for module_name in DEPENDENCY_NAMES:
    try:
        module = __import__(module_name)
        version = getattr(module, "__version__", None)
        if version is None and module_name == "yaml":
            version = getattr(module, "__with_libyaml__", "unknown")
        runtime_rows.append({"module": module_name, "available": True, "version": version})
    except Exception as exc:  # noqa: BLE001, PERF203 - optional dependency report should not stop the notebook.
        runtime_rows.append({"module": module_name, "available": False, "version": f"unavailable: {exc}"})

try:
    ppo_dependencies = experiments.training.ppo_tracking.detect_ppo_tracking_dependencies()
    ppo_runtime = experiments.training.ppo_tracking.detect_ppo_runtime_info()
except Exception as exc:  # noqa: BLE001
    ppo_dependencies = {"error": str(exc)}
    ppo_runtime = {"error": str(exc)}

show_table(runtime_rows, title="Import/runtime availability")
print("PPO dependency check:", json.dumps(ppo_dependencies, indent=2, sort_keys=True))
print("PPO runtime info:", json.dumps(ppo_runtime, indent=2, sort_keys=True))

In [ ]:
pyproject = REPO_ROOT / "pyproject.toml"
print("Dependency and package requirements source:", pyproject)
if pyproject.is_file():
    text = pyproject.read_text(encoding="utf-8")
    lines = text.splitlines()
    for index, line in enumerate(lines, start=1):
        if line.strip() == "dependencies = [":
            for dependency_line in lines[index - 1 :]:
                print(dependency_line)
                if dependency_line.strip() == "]":
                    break
            break
else:
    print("pyproject.toml not found.")

## Environment and Agent

The current environment is `envs.tracking_env.TrajectoryTrackingEnv`, a compact Gymnasium-compatible wrapper around `gym-pybullet-drones.envs.HoverAviary` with PID target-position actions.

**State / observation `S`:** the policy observes a 10-dimensional float vector:

- current XYZ position
- reference XYZ position
- XYZ position error, computed as current minus reference
- normalized trajectory progress in `[0, 1]`

**Action space `A`:** the underlying simulator action is a real PID target-position action with shape `(1, 3)`. PPO normally sees `NormalizedActionWrapper`, which maps continuous actions in `[-1, 1]` into the real task-local PID target-position bounds.

**Reward:** the implemented reward is a simple deterministic tracking objective:

- negative Euclidean position-error penalty
- negative squared-action-cost penalty
- tracking success flag when the position error is within the configured tolerance

**Episode diagnostics:** step info includes reference/current positions, tracking error, task shape, reference time/index, start-hold metadata, termination reason, base truncation causes, roll/pitch/yaw, velocity, angular velocity, and action metadata.


In [ ]:
env_summary = [
    {"item": "Environment class", "value": "envs.tracking_env.TrajectoryTrackingEnv"},
    {"item": "Observation dimensions", "value": envs.tracking_env.OBSERVATION_DIMENSIONS},
    {"item": "Observation fields", "value": "current_xyz, reference_xyz, error_xyz, progress"},
    {"item": "Real action interface", "value": "PID target-position action, shape (1, 3)"},
    {"item": "PPO-facing action interface", "value": "normalized Box([-1, 1]) through NormalizedActionWrapper"},
    {"item": "Reward config defaults", "value": asdict(envs.tracking_reward.TrackingRewardConfig())},
]
show_table(env_summary)

In [ ]:
if RUN_QUICK_DEMO:
    demo_task = {
        "task_type": validation.contracts.TASK_TYPE_TRAJECTORY,
        "shape": validation.contracts.SHAPE_LINE,
        "duration_sec": 3.0,
        "sample_rate_hz": 10.0,
        "start_hold_enabled": True,
        "start_hold_sec": 1.0,
        "exclude_start_hold_from_tracking_metrics": True,
        "start": [0.0, 0.0, 1.0],
        "end": [1.0, 0.0, 1.0],
    }
    reference = envs.task_adapter.make_task_reference(demo_task)
    obs = [
        *reference.positions[0].tolist(),
        *reference.positions[reference.tracking_phase_start_step].tolist(),
        *(reference.positions[0] - reference.positions[reference.tracking_phase_start_step]).tolist(),
        reference.tracking_phase_start_step / max(reference.positions.shape[0] - 1, 1),
    ]
    print("Validated demo task shape:", reference.shape)
    print("Reference samples:", reference.positions.shape[0])
    print("Tracking phase starts at step:", reference.tracking_phase_start_step)
    print("Example observation-like vector length:", len(obs))
else:
    print("RUN_QUICK_DEMO is False, so the lightweight reference demo was skipped.")

## Trajectory Tasks and Validation


Trajectory tasks are dictionaries with `task_type: trajectory`, a supported `shape`, and shape-specific parameters. Validation is deterministic and simulator-independent. It checks sampled references against arena, height, speed, acceleration, duration, and maximum adjacent-step-distance limits.

Start-hold behavior is part of the current task contract. Moving tasks can prepend a stationary phase at the first reference point. When `exclude_start_hold_from_tracking_metrics` is true, tracking-only metrics omit those held samples so evaluation focuses on path-following quality rather than initial stabilization.


In [ ]:
supported_shape_rows = [{"supported_shape": shape} for shape in validation.contracts.SUPPORTED_TRAJECTORY_SHAPES]
show_table(supported_shape_rows, title="Validation-supported trajectory shapes")

limits = validation.tasks.ValidationLimits()
limit_rows = [{"limit": key, "value": value} for key, value in asdict(limits).items()]
show_table(limit_rows, title="Default validation limits")

In [ ]:
task_config_path = REPO_ROOT / "configs/training/ppo_tracking_tasks.yaml"
task_config = load_yaml(task_config_path)
training_tasks = task_config.get("tasks", [])
validation_rows: list[dict[str, Any]] = []
for index, task in enumerate(training_tasks):
    result = validation.tasks.validate_task(task)
    validation_rows.append(
        {
            "index": index,
            "task_name": task.get("task_name", f"task_{index}"),
            "shape": task.get("shape"),
            "valid": result.is_valid,
            "messages": "; ".join(result.messages) if result.messages else "",
            "samples": None if result.trajectory is None else int(result.trajectory.positions.shape[0]),
            "start_hold_sec": result.start_hold_sec,
            "tracking_phase_start_step": result.tracking_phase_start_step,
            "exclude_start_hold": result.exclude_start_hold_from_tracking_metrics,
        }
    )
show_table(validation_rows, title=f"Validation results from {task_config_path.relative_to(REPO_ROOT)}")

In [ ]:
config_groups = {
    "direct_training": sorted((REPO_ROOT / "configs/training").glob("ppo_tracking_*.yaml")),
    "manual_curriculum": sorted((REPO_ROOT / "configs/curricula").glob("curriculum_manual_line_*.yaml")),
    "llm_curriculum": sorted((REPO_ROOT / "configs/curricula").glob("curriculum_llm*.yaml")),
    "evaluation_suites": sorted((REPO_ROOT / "configs/evaluation").glob("*_eval_suite.yaml")),
}
config_rows = [{"group": group, "path": str(config_path.relative_to(REPO_ROOT))} for group, paths in config_groups.items() for config_path in paths]
show_table(config_rows, title="Available task/training/curriculum/evaluation configs")

## Algorithm: PPO


The implemented RL algorithm is Proximal Policy Optimization (PPO) from Stable-Baselines3. PPO is a policy optimization method, not Q-learning. In this project, the policy class is `MlpPolicy`: a neural policy maps the compact observation vector to continuous normalized target-position actions.

For the handout parameter notation:

- `alpha` maps to PPO `learning_rate`
- `gamma` maps to PPO `gamma`

Current PPO configs expose `num_envs` as a top-level training setting. `num_envs: 1` is the normal single-environment path. Higher values parallelize PPO sample collection across vectorized environments, while PPO still interprets `ppo.n_steps` as the per-environment rollout length. The effective rollout size per PPO update is therefore `ppo.n_steps * num_envs`.

Deterministic evaluation remains separate from training parallelism: evaluation runs use policy predictions with deterministic actions and fixed evaluation steps. The current tiered configs vary training budget, `n_steps`, batch size, epochs, and `num_envs`, but the checked configs use the same `learning_rate` and `gamma`. Therefore the required alpha/gamma parameter study is not completed yet unless saved local runs with varied values are present.


In [ ]:
ppo_config_paths = [
    REPO_ROOT / "configs/training/ppo_tracking_smoke.yaml",
    REPO_ROOT / "configs/training/ppo_tracking_medium.yaml",
    REPO_ROOT / "configs/training/ppo_tracking_final.yaml",
]
ppo_rows: list[dict[str, Any]] = []
for config_path in ppo_config_paths:
    cfg = load_yaml(config_path)
    settings = experiments.training.ppo_config.load_ppo_config_from_mapping(cfg)
    resolved = settings.to_dict()
    num_envs = int(cfg.get("num_envs", 1))
    ppo_rows.append(
        {
            "config": str(config_path.relative_to(REPO_ROOT)),
            "run_name": cfg.get("run_name"),
            "task_index": cfg.get("task_index"),
            "total_timesteps": cfg.get("total_timesteps"),
            "eval_steps": cfg.get("eval_steps"),
            "num_envs": num_envs,
            "vec_env_type": "DummyVecEnv" if num_envs == 1 else "SubprocVecEnv",
            "effective_rollout_steps": settings.effective_rollout_steps(num_envs),
            "policy": resolved["policy"],
            "alpha_learning_rate": resolved["learning_rate"],
            "gamma": resolved["gamma"],
            "n_steps": resolved["n_steps"],
            "batch_size": resolved["batch_size"],
            "n_epochs": resolved["n_epochs"],
            "device": resolved["device"],
        }
    )
show_table(ppo_rows, title="Current PPO config values")

learning_rates = {row["alpha_learning_rate"] for row in ppo_rows}
gammas = {row["gamma"] for row in ppo_rows}
print("Alpha/gamma varied across checked configs:", len(learning_rates) > 1 or len(gammas) > 1)
print("Configured num_envs by tier:", {row["config"]: row["num_envs"] for row in ppo_rows})

## Training Setup


Training commands are intentionally shown rather than executed by default. Smoke runs are for correctness checks. Medium and final tiers are meaningful training runs and should be launched intentionally outside the report notebook.

Direct PPO training uses top-level `num_envs`. The current configs use `num_envs: 1` for smoke, `num_envs: 4` for medium, and `num_envs: 8` for final. Higher values parallelize training-time sample collection only; deterministic evaluation remains a separate single-rollout protocol.

The LLM curriculum commands are also shown rather than executed. The mock-provider smoke config can run offline and supports `--dry-run-proposals` to exercise proposal parsing, repair, validation, and JSONL logging without starting PPO training. The local LLM config expects an external OpenAI-compatible HTTP server and does not start that server from the repository.


In [ ]:
def command(*parts: str) -> str:
    """Return one shell command string for display only."""
    return " ".join(parts)


training_commands = [
    command(
        "python -m src.experiments.cli.experiments_cli_train_tracking",
        "--config configs/training/ppo_tracking_smoke.yaml",
        "--run-name direct_ppo_line_smoke_seed0",
        "--seed 0",
        "--wandb-mode disabled",
    ),
    command(
        "python -m src.experiments.cli.experiments_cli_train_curriculum",
        "--config configs/curricula/curriculum_manual_line_smoke.yaml",
        "--seed 0",
        "--wandb-mode disabled",
    ),
    command(
        "python -m src.experiments.cli.experiments_cli_train_llm_curriculum",
        "--config configs/curricula/curriculum_llm_smoke.yaml",
        "--dry-run-proposals",
        "--wandb-mode disabled",
    ),
    command(
        "python -m src.experiments.cli.experiments_cli_train_llm_curriculum",
        "--config configs/curricula/curriculum_llm_smoke.yaml",
        "--seed 0",
        "--wandb-mode disabled",
    ),
    command(
        "python -m src.experiments.cli.experiments_cli_train_llm_curriculum",
        "--config configs/curricula/curriculum_llm_local_smoke.yaml",
        "--provider openai_compatible",
        "--api-base http://127.0.0.1:18080/v1",
        "--dry-run-proposals",
        "--wandb-mode disabled",
    ),
    command(
        "python -m src.experiments.cli.experiments_cli_train_tracking",
        "--config configs/training/ppo_tracking_medium.yaml",
        "--run-name direct_ppo_line_medium_seed0",
        "--seed 0",
        "--wandb-mode offline",
    ),
    command(
        "python -m src.experiments.cli.experiments_cli_train_curriculum",
        "--config configs/curricula/curriculum_manual_line_medium.yaml",
        "--seed 0",
        "--wandb-mode offline",
    ),
    command(
        "python -m src.experiments.cli.experiments_cli_train_tracking",
        "--config configs/training/ppo_tracking_final.yaml",
        "--run-name direct_ppo_line_final_seed0",
        "--seed 0",
        "--wandb-mode auto",
    ),
    command(
        "python -m src.experiments.cli.experiments_cli_train_curriculum",
        "--config configs/curricula/curriculum_manual_line_final.yaml",
        "--seed 0",
        "--wandb-mode auto",
    ),
]
for item in training_commands:
    print(item)

if TRAIN_FROM_SCRATCH:
    print("\nTRAIN_FROM_SCRATCH=True, but this report cell still does not launch long training automatically.")
    print("Run the smoke commands manually if a fresh correctness run is desired.")
else:
    print("\nTRAIN_FROM_SCRATCH=False, so no training is started by this notebook.")

In [ ]:
curriculum_rows: list[dict[str, Any]] = []
for config_path in sorted((REPO_ROOT / "configs/curricula").glob("curriculum_manual_line_*.yaml")):
    cfg = load_yaml(config_path)
    stages = cfg.get("stages", [])
    curriculum_rows.append(
        {
            "config": str(config_path.relative_to(REPO_ROOT)),
            "curriculum_name": cfg.get("curriculum_name"),
            "base_training_config": cfg.get("base_training_config"),
            "stage_count": len(stages),
            "total_stage_timesteps": sum(int(stage.get("total_timesteps", 0)) for stage in stages if isinstance(stage, dict)),
            "stage_names": ", ".join(str(stage.get("stage_name")) for stage in stages if isinstance(stage, dict)),
        }
    )
show_table(curriculum_rows, title="Manual curriculum tiers")

## LLM Curriculum Task Proposal Pipeline

Phase 6 is implemented on this branch. The LLM remains a curriculum task proposer, not a drone controller and not a generator of executable Python code. The proposal pipeline asks for exactly one JSON object, rejects markdown/prose/code/unsupported keys, normalizes the task through the task schema, and validates the resulting trajectory with deterministic validation before any training stage can use it.

Invalid proposals can trigger bounded repair attempts. Every proposal attempt is logged as JSONL under the run-scoped `llm_logs/proposals.jsonl` path. The repository includes a deterministic mock provider for offline smoke testing and an OpenAI-compatible local HTTP provider for use with an external local model server. Real LLM curriculum training results should only be claimed when the corresponding saved run artifacts exist.


In [ ]:
llm_config_paths = [
    REPO_ROOT / "configs/curricula/curriculum_llm_smoke.yaml",
    REPO_ROOT / "configs/curricula/curriculum_llm_local_smoke.yaml",
]
llm_rows: list[dict[str, Any]] = []
for config_path in llm_config_paths:
    if not config_path.is_file():
        continue
    settings = experiments.curriculum.llm_training.load_llm_curriculum_settings(config_path)
    bootstrap_name = None if settings.bootstrap_stage is None else settings.bootstrap_stage.stage_name
    llm_rows.append(
        {
            "config": str(config_path.relative_to(REPO_ROOT)),
            "curriculum_name": settings.curriculum_name,
            "provider": settings.llm_provider,
            "model": settings.llm_model,
            "max_stages": settings.max_stages,
            "bootstrap_stage": bootstrap_name,
            "max_repair_attempts": settings.proposal_settings.max_repair_attempts,
            "skip_invalid_proposals": settings.proposal_settings.skip_invalid_proposals,
            "recent_context_limit": settings.proposal_settings.recent_context_limit,
            "stage_total_timesteps": settings.stage_total_timesteps,
            "stage_eval_steps": settings.stage_eval_steps,
        }
    )
show_table(llm_rows, title="LLM curriculum configs")

schema = llm.task_schema.build_task_schema()
schema_rows = [
    {"schema_item": "supported_shapes", "value": ", ".join(schema["shapes"])},
    {"schema_item": "required_fields", "value": ", ".join(schema["required_fields"])},
    {"schema_item": "optional_metadata_fields", "value": ", ".join(schema["optional_metadata_fields"])},
    {"schema_item": "forbidden_example_fields", "value": ", ".join(schema["forbidden_example_fields"])},
]
show_table(schema_rows, title="LLM proposal schema summary")
print("Supported LLM providers:", ", ".join(llm.client.SUPPORTED_PROVIDERS))
print("Proposal stats keys:", ", ".join(llm.curriculum.empty_proposal_stats().keys()))

## Evaluation Protocol


Evaluation suites are YAML configs loaded through `experiments.evaluation.suites`. The current canonical suites are:

- `line_eval_suite.yaml`: focused one-meter line evaluation
- `final_benchmark_eval_suite.yaml`: final line benchmark variants
- `generalization_eval_suite.yaml`: validation-supported non-line tasks such as polyline, circle, and vertical paths

Metrics include mean/final/max position error, tracking-only error when start-hold is excluded, reward, XY span and tracking ratio, action saturation, termination/truncation counts, failure modes, traces, plots, GIF renders, and diagnostics.


In [ ]:
suite_paths = [
    REPO_ROOT / "configs/evaluation/line_eval_suite.yaml",
    REPO_ROOT / "configs/evaluation/final_benchmark_eval_suite.yaml",
    REPO_ROOT / "configs/evaluation/generalization_eval_suite.yaml",
]

suite_rows: list[dict[str, Any]] = []
for suite_path in suite_paths:
    suite = experiments.evaluation.suites.load_evaluation_suite(suite_path)
    suite_rows.extend(
        {
            "suite_config": str(suite_path.relative_to(REPO_ROOT)),
            "evaluation_name": suite.evaluation_name,
            "eval_steps": suite.eval_steps,
            "task_name": task.task_name,
            "task_shape": task.task_shape,
            "render_enabled": suite.render.enabled,
            "plots_enabled": suite.plots.enabled,
            "traces_enabled": suite.traces.enabled,
        }
        for task in suite.tasks
    )
show_table(suite_rows, title="Evaluation suite tasks")

## Saved Results Loader


In [ ]:
def safe_load_json(path: Path) -> dict[str, Any] | list[Any] | None:
    """Load JSON and return None on malformed or missing files."""
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:  # noqa: BLE001
        print(f"Could not load {path}: {exc}")
        return None


def safe_load_jsonl(path: Path) -> list[dict[str, Any]] | None:
    """Load a proposal JSONL log and return None on malformed or missing files."""
    try:
        return llm.logging.read_jsonl(path)
    except Exception as exc:  # noqa: BLE001
        print(f"Could not load {path}: {exc}")
        return None


def rel_to_storage(path: Path) -> str:
    """Return a compact path relative to the selected storage root when possible."""
    try:
        return str(path.resolve(strict=False).relative_to(STORAGE_ROOT.resolve(strict=False)))
    except ValueError:
        return str(path)


def collect_files(patterns: list[str]) -> list[Path]:
    """Collect matching files below RUNS_ROOT without failing when storage is absent."""
    if not RUNS_ROOT.is_dir():
        return []
    files: list[Path] = []
    for pattern in patterns:
        files.extend(RUNS_ROOT.glob(pattern))
    return sorted(set(files))


manifest_files = collect_files(["*/run_manifest.json"])
training_metric_files = collect_files(
    [
        "*/training/metrics/*.json",
        "*/stages/stage*/training/metrics/*.json",
    ]
)
evaluation_metric_files = collect_files(
    [
        "*/evaluations/*/metrics/*.json",
        "*/evaluations/*/*/metrics/*.json",
        "*/stages/stage*/evaluations/*/metrics/*.json",
        "*/stages/stage*/evaluations/*/*/metrics/*.json",
    ]
)
llm_proposal_log_files = collect_files(["*/llm_logs/*.jsonl"])
png_files = collect_files(
    [
        "*/evaluations/**/*.png",
        "*/stages/stage*/evaluations/**/*.png",
    ]
)
gif_files = collect_files(
    [
        "*/evaluations/**/*.gif",
        "*/stages/stage*/evaluations/**/*.gif",
    ]
)

manifest_payloads = [(path, safe_load_json(path)) for path in manifest_files]
training_metrics = [(path, safe_load_json(path)) for path in training_metric_files]
evaluation_metrics = [(path, safe_load_json(path)) for path in evaluation_metric_files]
proposal_log_payloads = [(path, safe_load_jsonl(path)) for path in llm_proposal_log_files]

result_counts = [
    {"artifact_type": "run manifests", "count": len(manifest_files)},
    {"artifact_type": "training metric JSON", "count": len(training_metric_files)},
    {"artifact_type": "evaluation metric JSON", "count": len(evaluation_metric_files)},
    {"artifact_type": "LLM proposal JSONL logs", "count": len(llm_proposal_log_files)},
    {"artifact_type": "plot PNG files", "count": len(png_files)},
    {"artifact_type": "GIF files", "count": len(gif_files)},
]
print("Selected RUNS_ROOT:", RUNS_ROOT)
show_table(result_counts, title="Saved local result artifacts found")

In [ ]:
SUMMARY_METRIC_KEYS = [
    "mean_position_error_m",
    "mean_position_error_tracking_m",
    "final_position_error_m",
    "max_position_error_m",
    "reward_mean",
    "total_reward",
    "actual_xy_span_m",
    "reference_xy_span_m",
    "xy_tracking_ratio",
    "action_saturation_fraction",
    "real_action_saturation_fraction",
    "eval_terminated_count",
    "eval_truncated_count",
    "failure_primary_mode",
    "failure_overall_status",
]


PROPOSAL_STAT_KEYS = [
    "total_proposals",
    "invalid_proposals",
    "repair_attempts",
    "repair_successes",
    "final_accepted_tasks",
]


def add_proposal_stats(row: dict[str, Any], payload: dict[str, Any]) -> None:
    """Copy proposal statistics from a manifest-like payload into a display row."""
    proposal_stats = payload.get("proposal_stats")
    if not isinstance(proposal_stats, dict):
        return
    for key in PROPOSAL_STAT_KEYS:
        row[f"proposal_{key}"] = proposal_stats.get(key)


def metric_summary_row(path: Path, payload: Any, artifact_kind: str) -> dict[str, Any]:
    """Build a compact summary row from one metrics payload."""
    if not isinstance(payload, dict):
        return {"artifact_kind": artifact_kind, "path": rel_to_storage(path), "loaded": False}
    row: dict[str, Any] = {
        "artifact_kind": artifact_kind,
        "path": rel_to_storage(path),
        "run_name": payload.get("run_name") or payload.get("training_run_name"),
        "run_kind": payload.get("run_kind"),
        "curriculum_kind": payload.get("curriculum_kind"),
        "mode": payload.get("mode"),
        "task_shape": payload.get("task_shape") or payload.get("task_shape_used_for_evaluation"),
        "suite_task_name": payload.get("suite_task_name"),
        "evaluation_name": payload.get("evaluation_name"),
        "total_timesteps": payload.get("total_timesteps"),
        "seed": payload.get("seed"),
        "num_envs": payload.get("num_envs"),
        "vec_env_type": payload.get("vec_env_type"),
        "effective_rollout_steps": payload.get("effective_rollout_steps"),
        "dry_run_proposals": payload.get("dry_run_proposals"),
        "llm_provider": payload.get("llm_provider"),
    }
    for key in SUMMARY_METRIC_KEYS:
        if key in payload:
            row[key] = payload.get(key)
    ppo_payload = payload.get("ppo_config")
    if isinstance(ppo_payload, dict):
        row["alpha_learning_rate"] = ppo_payload.get("learning_rate")
        row["gamma"] = ppo_payload.get("gamma")
    add_proposal_stats(row, payload)
    return row


def proposal_log_summary_row(path: Path, events: list[dict[str, Any]] | None) -> dict[str, Any]:
    """Build a compact row from one LLM proposal JSONL log."""
    if events is None:
        return {"path": rel_to_storage(path), "loaded": False}
    return {
        "path": rel_to_storage(path),
        "event_count": len(events),
        "accepted_events": sum(1 for event in events if event.get("status") == "accepted"),
        "rejected_events": sum(1 for event in events if event.get("status") == "rejected"),
        "repair_attempt_events": sum(1 for event in events if event.get("is_repair_attempt") is True),
        "invalid_validation_events": sum(1 for event in events if event.get("validation_status") == "invalid"),
    }


training_rows = [metric_summary_row(path, payload, "training") for path, payload in training_metrics]
evaluation_rows = [metric_summary_row(path, payload, "evaluation") for path, payload in evaluation_metrics]
proposal_log_rows = [proposal_log_summary_row(path, payload) for path, payload in proposal_log_payloads]
manifest_rows = []
for manifest_path, payload in manifest_payloads:
    if isinstance(payload, dict):
        config_section = payload.get("config") if isinstance(payload.get("config"), dict) else {}
        manifest_row = {
            "path": rel_to_storage(manifest_path),
            "run_name": payload.get("run_name"),
            "run_kind": payload.get("run_kind"),
            "curriculum_kind": payload.get("curriculum_kind"),
            "mode": payload.get("mode"),
            "total_timesteps": payload.get("total_timesteps"),
            "seed": payload.get("seed"),
            "num_envs": payload.get("num_envs"),
            "vec_env_type": payload.get("vec_env_type"),
            "effective_rollout_steps": payload.get("effective_rollout_steps"),
            "stage_count": payload.get("stage_count"),
            "dry_run_proposals": payload.get("dry_run_proposals"),
            "llm_provider": payload.get("llm_provider") or config_section.get("llm_provider"),
            "llm_model": payload.get("llm_model") or config_section.get("llm_model"),
            "final_stage_run_name": payload.get("final_stage_run_name"),
            "final_model_path": payload.get("final_model_path"),
            "proposal_log_path": payload.get("proposal_log_path"),
            "evaluation_index_entries": (payload.get("evaluation_index") or {}).get("entry_count")
            if isinstance(payload.get("evaluation_index"), dict)
            else None,
        }
        add_proposal_stats(manifest_row, payload)
        manifest_rows.append(manifest_row)
    else:
        manifest_rows.append({"path": rel_to_storage(manifest_path), "loaded": False})

llm_manifest_rows = [row for row in manifest_rows if row.get("curriculum_kind") == "llm" or row.get("mode") == "llm_curriculum"]
llm_completed_training_manifest_rows = [row for row in llm_manifest_rows if row.get("dry_run_proposals") is False and row.get("final_model_path")]

show_table(manifest_rows, title="Run manifests")
show_table(training_rows, title="Training metrics")
show_table(evaluation_rows, title="Evaluation metrics")
show_table(proposal_log_rows, title="LLM proposal logs")

## Results So Far


The cells below display only artifacts that exist in local storage. External storage may contain runs produced by earlier branches or earlier experiments, so paths, manifests, configs, and timestamps are the provenance source. Smoke runs, dry-run proposal logs, and locally available artifacts are useful implementation evidence but are not automatically final report results.

Missing artifacts are reported as missing rather than filled with synthetic data.


In [ ]:
if not USE_SAVED_RESULTS:
    print("USE_SAVED_RESULTS=False, so saved-result display is disabled.")
elif not RUNS_ROOT.is_dir():
    print("No saved report-ready results found locally yet: RUNS_ROOT does not exist.")
    print("Expected location:", RUNS_ROOT)
elif not training_rows and not evaluation_rows and not proposal_log_rows:
    print("No saved report-ready metrics or LLM proposal logs found locally yet.")
else:
    print(
        f"Loaded {len(training_rows)} training metric file(s), "
        f"{len(evaluation_rows)} evaluation metric file(s), and "
        f"{len(proposal_log_rows)} LLM proposal log summary row(s)."
    )
    if llm_manifest_rows:
        print("Local LLM curriculum manifest(s) found. Treat them as local artifacts unless selected for the final report.")
    if not llm_completed_training_manifest_rows:
        print("No completed local LLM curriculum training manifest with a final model was found.")
    selected_columns = [
        "artifact_kind",
        "run_name",
        "run_kind",
        "curriculum_kind",
        "mode",
        "task_shape",
        "suite_task_name",
        "total_timesteps",
        "num_envs",
        "vec_env_type",
        "effective_rollout_steps",
        "mean_position_error_m",
        "mean_position_error_tracking_m",
        "final_position_error_m",
        "max_position_error_m",
        "xy_tracking_ratio",
        "failure_primary_mode",
    ]
    show_table(training_rows[:20] + evaluation_rows[:20], columns=selected_columns, title="Compact metric summary")

In [ ]:
if USE_SAVED_RESULTS and pd is not None and training_rows:
    import matplotlib.pyplot as plt

    df_train = pd.DataFrame(training_rows)
    plot_columns = [column for column in ["mean_position_error_m", "mean_position_error_tracking_m"] if column in df_train.columns]
    if plot_columns and "total_timesteps" in df_train.columns:
        plot_df = df_train.dropna(subset=["total_timesteps"]).copy()
        if not plot_df.empty:
            fig, axis = plt.subplots(figsize=(8, 4))
            for column in plot_columns:
                axis.scatter(plot_df["total_timesteps"], plot_df[column], label=column)
            axis.set_xlabel("total timesteps")
            axis.set_ylabel("position error m")
            axis.set_title("Available scalar training metrics, not a full learning curve")
            axis.grid(True, alpha=0.3)
            axis.legend()
            plt.show()
        else:
            print("Training metrics exist, but none include total_timesteps for plotting.")
    else:
        print("Training metrics do not include enough scalar error fields for a plot.")
elif USE_SAVED_RESULTS and training_rows:
    print("pandas/matplotlib display is unavailable; see the metric tables above.")
else:
    print("No training metric rows available for plotting.")

In [ ]:
if USE_SAVED_RESULTS and png_files:
    print(f"Found {len(png_files)} plot PNG file(s). Displaying up to 6.")
    for plot_path in png_files[:6]:
        print(rel_to_storage(plot_path))
        if Image is not None:
            ipython_display(Image(filename=str(plot_path)))
else:
    print("No plot PNG files found locally yet.")

if USE_SAVED_RESULTS and gif_files:
    print(f"Found {len(gif_files)} GIF file(s). Listing all and displaying the first when possible.")
    for gif_path in gif_files:
        print(rel_to_storage(gif_path))
    if Image is not None:
        ipython_display(Image(filename=str(gif_files[0])))
else:
    print("No GIF files found locally yet.")

In [ ]:
if not training_rows and not evaluation_rows:
    print("Commands to generate local artifacts:")
    for item in training_commands[:5]:
        print(item)
    print(
        command(
            "python -m src.experiments.cli.experiments_cli_evaluate_policy",
            "--run-manifest storage/runs/direct_ppo_line_smoke_seed0/run_manifest.json",
            "--wandb-mode disabled",
        )
    )
    print(
        command(
            "python -m src.experiments.cli.experiments_cli_evaluate_curriculum",
            "--summary storage/runs/curriculum_manual_line_smoke_seed0/run_manifest.json",
            "--wandb-mode disabled",
        )
    )
else:
    print("Saved artifacts were found. Use the manifest paths above to verify which runs are current and report-ready.")

## Requirement Status / Missing According to PDF


In [ ]:
def has_varied_alpha_gamma(rows: list[dict[str, Any]]) -> bool:
    """Return whether loaded training rows include more than one alpha or gamma value."""
    alphas = {row.get("alpha_learning_rate") for row in rows if row.get("alpha_learning_rate") is not None}
    gammas = {row.get("gamma") for row in rows if row.get("gamma") is not None}
    return len(alphas) > 1 or len(gammas) > 1


def has_repeated_seed_evidence(rows: list[dict[str, Any]]) -> bool:
    """Return whether metrics include at least two seeds for one setup."""
    seeds_by_setup: dict[tuple[Any, Any, Any], set[Any]] = {}
    for row in rows:
        seed = row.get("seed")
        if seed is None:
            continue
        key = (row.get("run_kind"), row.get("mode"), row.get("task_shape"))
        seeds_by_setup.setdefault(key, set()).add(seed)
    return any(len(seeds) > 1 for seeds in seeds_by_setup.values())


def row_text(row: dict[str, Any], keys: tuple[str, ...]) -> str:
    """Return lowercase searchable row text for status classification."""
    return " ".join(str(row.get(key) or "") for key in keys).lower()


all_result_rows = training_rows + evaluation_rows + manifest_rows

direct_results = [row for row in all_result_rows if row.get("run_kind") == "direct_ppo" or str(row.get("run_name") or "").startswith("direct_ppo")]
manual_results = [
    row
    for row in all_result_rows
    if "curriculum_manual" in row_text(row, ("run_name", "path")) or row.get("curriculum_kind") in {"manual", "manual_curriculum"}
]
llm_loop_present = all(
    (REPO_ROOT / path).exists()
    for path in [
        "src/llm/llm_curriculum.py",
        "src/llm/llm_client.py",
        "src/experiments/curriculum/experiments_curriculum_llm_training.py",
        "src/experiments/cli/experiments_cli_train_llm_curriculum.py",
        "configs/curricula/curriculum_llm_smoke.yaml",
        "configs/curricula/curriculum_llm_local_smoke.yaml",
    ]
)
multi_env_present = any(row.get("num_envs") is not None for row in ppo_rows)
phase5_present = any(
    (REPO_ROOT / path).exists()
    for path in [
        "src/experiments/comparison",
        "src/experiments/cli/experiments_cli_compare.py",
        "src/experiments/cli/experiments_cli_compare_runs.py",
    ]
)
proposal_stats_available = any(row.get("proposal_total_proposals") not in (None, 0) for row in manifest_rows) or any(
    row.get("event_count", 0) > 0 for row in proposal_log_rows
)

if llm_completed_training_manifest_rows:
    llm_status = "implemented; completed local LLM curriculum artifact(s) found, not automatically final report results"
elif llm_manifest_rows or proposal_log_rows:
    llm_status = "implemented; local dry-run/proposal artifacts found, final LLM training results not available yet"
else:
    llm_status = "implemented; no local LLM curriculum result artifacts found yet"

requirement_rows = [
    {
        "Requirement": "Header",
        "Current notebook/repo status": "present after this notebook update; local OST logo path referenced",
        "Evidence/source in repo": "notebook header; docs/images/OST_Logo_DE_RGB@2000ppi.png",
        "Missing action": "Ensure the logo asset is included with the final submitted/exported report.",
    },
    {
        "Requirement": "Introduction",
        "Current notebook/repo status": "present after this notebook update",
        "Evidence/source in repo": "notebook Introduction section",
        "Missing action": "",
    },
    {
        "Requirement": "Objective/approach",
        "Current notebook/repo status": "present after this notebook update",
        "Evidence/source in repo": "PROJECT_BRIEF.md; notebook objective section",
        "Missing action": "",
    },
    {
        "Requirement": "Environment/agent",
        "Current notebook/repo status": "present after this notebook update",
        "Evidence/source in repo": "src/envs/envs_tracking_env.py; src/envs/envs_tracking_reward.py",
        "Missing action": "",
    },
    {
        "Requirement": "Algorithm",
        "Current notebook/repo status": "present after this notebook update, including num_envs rollout semantics",
        "Evidence/source in repo": "configs/training/ppo_tracking_*.yaml; src/experiments/training/experiments_training_ppo_config.py",
        "Missing action": "",
    },
    {
        "Requirement": "Train/test learning curves",
        "Current notebook/repo status": "partial/local artifacts found" if training_rows or png_files else "missing",
        "Evidence/source in repo": f"{len(training_rows)} training metric files; {len(png_files)} PNG files under selected storage",
        "Missing action": "Run selected report-ready training/evaluation and save learning-curve artifacts.",
    },
    {
        "Requirement": "Different alpha/gamma parameter sets",
        "Current notebook/repo status": "present" if has_varied_alpha_gamma(training_rows) else "missing",
        "Evidence/source in repo": "loaded completed training metrics; PPO YAML configs currently share learning_rate and gamma",
        "Missing action": "Run controlled parameter study varying learning_rate and/or gamma.",
    },
    {
        "Requirement": "Statistical evaluation",
        "Current notebook/repo status": "present" if has_repeated_seed_evidence(training_rows + evaluation_rows) else "missing",
        "Evidence/source in repo": "loaded metric seeds",
        "Missing action": "Run repeated seeds and add statistical summaries/tests.",
    },
    {
        "Requirement": "Direct PPO baseline",
        "Current notebook/repo status": "implemented/configured; local artifacts found"
        if direct_results
        else "implemented/configured; no local results found",
        "Evidence/source in repo": "configs/training/ppo_tracking_*.yaml; src/experiments/training",
        "Missing action": "Run/evaluate final baseline if final artifacts are missing.",
    },
    {
        "Requirement": "Manual curriculum",
        "Current notebook/repo status": "implemented/configured; local artifacts found"
        if manual_results
        else "implemented/configured; no local results found",
        "Evidence/source in repo": "configs/curricula/curriculum_manual_line_*.yaml; src/experiments/curriculum",
        "Missing action": "Run/evaluate final manual curriculum if final artifacts are missing.",
    },
    {
        "Requirement": "LLM-guided curriculum",
        "Current notebook/repo status": llm_status if llm_loop_present else "not implemented yet; schema helpers only",
        "Evidence/source in repo": "src/llm/*; LLM curriculum training module; curriculum_llm*.yaml",
        "Missing action": "Run/evaluate selected LLM curriculum experiments before making performance claims.",
    },
    {
        "Requirement": "Invalid LLM proposal rate / repair success rate",
        "Current notebook/repo status": "local proposal logs/stats available"
        if proposal_stats_available
        else "implemented accounting; report metric missing until selected LLM logs exist",
        "Evidence/source in repo": "llm_logs/proposals.jsonl and manifest proposal_stats when LLM curriculum runs exist",
        "Missing action": "Aggregate proposal/repair rates from the selected report LLM runs.",
    },
    {
        "Requirement": "Multi-env PPO",
        "Current notebook/repo status": "implemented; top-level num_envs in current PPO configs" if multi_env_present else "missing",
        "Evidence/source in repo": "configs/training/ppo_tracking_*.yaml; PPOConfig.effective_rollout_steps(num_envs)",
        "Missing action": "",
    },
    {
        "Requirement": "Phase 5 comparison pipeline",
        "Current notebook/repo status": "implemented" if phase5_present else "not implemented yet",
        "Evidence/source in repo": "FINAL_CLEANUP_PLAN.md roadmap; no comparison package/CLI found"
        if not phase5_present
        else "comparison package/CLI found",
        "Missing action": "Implement comparison pipeline after matching direct/manual/LLM suite evaluations exist.",
    },
    {
        "Requirement": "References",
        "Current notebook/repo status": "present after this notebook update",
        "Evidence/source in repo": "notebook References section; README references",
        "Missing action": "",
    },
    {
        "Requirement": "Declaration of independence",
        "Current notebook/repo status": "present after this notebook update",
        "Evidence/source in repo": "DRL_Project.pdf extracted declaration text",
        "Missing action": "Add signature/date if a submitted PDF export requires it.",
    },
    {
        "Requirement": "Dependency/package requirements",
        "Current notebook/repo status": "present by reference",
        "Evidence/source in repo": "pyproject.toml and uv.lock",
        "Missing action": "",
    },
]
show_table(requirement_rows)

## Discussion

The current repository supports a clean single-drone tracking workflow: deterministic trajectory generation and validation, a compact tracking environment, PPO training through Stable-Baselines3, configurable training-time vectorized sample collection through `num_envs`, fixed manual curriculum training, Phase 6 LLM-guided curriculum proposal/training orchestration, and config-driven evaluation suites with canonical artifact paths under `storage/runs/<run_id>/`.

Conclusions about learning performance require completed saved runs. If local storage contains only smoke runs or dry-run proposal logs, those prove execution and artifact plumbing, not final research performance. If final or medium runs are missing, the notebook should be treated as a report scaffold plus provenance loader rather than as a completed experimental comparison.

The current action abstraction is intentionally high level: PPO predicts normalized target-position commands that are mapped to PID target-position bounds. This is a useful and stable learning interface, but it is not direct motor/RPM control. It hides some low-level flight dynamics behind the simulator PID controller.

The current validation-supported shapes are those listed by `validation.contracts.SUPPORTED_TRAJECTORY_SHAPES`. Future trajectory families such as figure-eight, star, or spiral should not be claimed as supported until validation contracts and task builders include them.

No LLM performance claim should be made merely because Phase 6 exists. The implementation can generate, repair, validate, log, and train from LLM proposals, but usefulness must be judged from real saved LLM curriculum runs evaluated against the same suites as direct PPO and manual curriculum. Invalid proposal rate and repair success rate should be computed from selected `llm_logs/proposals.jsonl` files or manifest `proposal_stats`, not guessed.


## Conclusion and Outlook

The project is in a strong integration state for direct PPO, manual curriculum, Multi-Env PPO rollout collection, and the Phase 6 LLM curriculum pipeline. The notebook can execute as a report, inspect current configs, validate tasks, summarize LLM proposal contracts, and load whatever saved artifacts exist locally without retraining by default.

Next steps:

1. Run selected report-ready direct PPO, manual curriculum, and LLM curriculum experiments without inventing missing results.
2. Implement Phase 5 comparison once runs own evaluations for the same suite.
3. Refresh this notebook after Phase 5 and stable report-ready results exist.
4. Add repeated seeds and statistical tests for final claims.
5. Consider SAC or TD3 comparisons, richer reward shaping, more validation-supported task families, and direct motor/RPM action spaces only as scoped future work.
6. Keep multi-drone behavior as a showcase unless a full multi-agent formulation is added.


## References

- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., and Klimov, O. (2017). *Proximal Policy Optimization Algorithms*.
- Raffin, A., Hill, A., Gleave, A., Kanervisto, A., Ernestus, M., and Dormann, N. (2021). *Stable-Baselines3: Reliable Reinforcement Learning Implementations*. Journal of Machine Learning Research.
- Panerati, J., Zheng, H., Zhou, S., Xu, J., Prorok, A., and Schoellig, A. P. (2021). *Learning to Fly: A Gym Environment with PyBullet Physics for Reinforcement Learning of Multi-agent Quadcopter Control*.
- Coumans, E. and Bai, Y. (2016-2021). *PyBullet: A Python Module for Physics Simulation for Games, Robotics and Machine Learning*.
- Farama Foundation. *Gymnasium documentation*.


## Declaration of Independence

I hereby certify that I have written this thesis independently and have not used any auxiliary materials other than those indicated. The passages of the work, which are taken in the wording or the sense after other works (to it also Internet sources count), were marked under indication of the source.

<table style="width:100%; background-color: white; padding: 10px; border-radius: 6px; box-shadow: 0 0 5px rgba(0,0,0,0.2); margin-top:20px;">
  <tr>
    <td align="left">
      <img src="docs/images/Unterschrift.png" alt="Unterschrift" style="height:80px;">
    </td>
  </tr>
</table>